<a href="https://colab.research.google.com/github/ArnavP2305/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv("content_refresh_anonymized.csv")

print(df.shape)
print(df.columns.tolist())

df.head()

(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

# My Rule

The goal is to identify content pages that are good candidates for content refresh.

Rule:

- The page has not been updated for at least 180 days.
- The page has received at least 500 impressions.
- The average search position is greater than 10.

If all three conditions are satisfied, the page will be recommended for content refresh.

# Reason Code

stale_visible_lowrank

# Action

refresh_content

In [2]:
# Signal 1: Days Since Last Update

bins = [0, 90, 180, 365, df["days_since_last_update"].max() + 1]
labels = ["0-90", "91-180", "181-365", "365+"]

signal1 = df.copy()
signal1["update_bucket"] = pd.cut(
    signal1["days_since_last_update"],
    bins=bins,
    labels=labels,
    right=False
)

table1 = (
    signal1.groupby("update_bucket")
    .agg(
        n=("content_id", "count"),
        avg_impressions=("impressions_90d", "mean"),
        avg_position=("avg_position", "mean"),
    )
)

print(table1)

                   n  avg_impressions  avg_position
update_bucket                                      
0-90           20655      4219.161317     15.692394
91-180          9171      7486.665140     17.901461
181-365          169      1206.893491     11.169822
365+               5         8.200000     16.600000


/tmp/ipykernel_1582/2222581509.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal1.groupby("update_bucket")


# Signal 1: Days Since Last Update

**Hypothesis:** Older pages are more likely to require a content refresh.

| Bucket | n | Avg Impressions | Avg Position |
|--------|----:|---------------:|-------------:|
|0–90|20655|4219.16|15.69|
|91–180|9171|7486.67|17.90|
|181–365|169|1206.89|11.17|
|365+|5|8.20|16.60|

**Verdict:** MIXED

The relationship is not consistently increasing with age. The oldest buckets contain very few pages, so there is not enough evidence to conclude that older content always performs worse. Age alone should not determine the refresh decision.

In [3]:
# Signal 2: Average Position

bins = [0, 5, 10, 20, 100]
labels = ["1-5", "6-10", "11-20", "20+"]

signal2 = df.copy()

signal2["position_bucket"] = pd.cut(
    signal2["avg_position"],
    bins=bins,
    labels=labels,
    right=False
)

table2 = (
    signal2.groupby("position_bucket")
    .agg(
        n=("content_id", "count"),
        avg_impressions=("impressions_90d", "mean"),
        avg_ctr=("ctr", "mean"),
    )
)

print(table2)

                    n  avg_impressions   avg_ctr
position_bucket                                 
1-5              4879      7385.844845  1.301629
6-10             9159      6583.323070  0.522342
11-20            7366      3149.190334  0.322224
20+              8581      4251.465214  0.211365


/tmp/ipykernel_1582/3956267692.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal2.groupby("position_bucket")


# Signal 2: Average Position

**Hypothesis:** Pages with worse average positions generally have lower CTR.

| Position Bucket | n | Avg Impressions | Avg CTR |
|-----------------|---:|----------------:|---------:|
|1–5|4879|7385.84|1.30|
|6–10|9159|6583.32|0.52|
|11–20|7366|3149.19|0.32|
|20+|8581|4251.47|0.21|

**Verdict:** CONFIRMED

The results show a clear decline in CTR as average position becomes worse. This confirms that search position is an important signal and should be included in the baseline rule.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# Rule conditions
stale = df["days_since_last_update"] >= 180
visible = df["impressions_90d"] >= 500
low_rank = df["avg_position"] > 10

# Baseline score
df["score"] = np.where(
    stale & visible & low_rank,
    df["impressions_90d"],
    0
)

In [5]:
df["reason_code"] = np.where(
    df["score"] > 0,
    "stale_visible_lowrank",
    "no_action"
)

In [6]:
df["action"] = np.where(
    df["score"] > 0,
    "refresh_content",
    "keep_monitoring"
)

In [7]:
ranked_df = df.sort_values(
    by="score",
    ascending=False
).reset_index(drop=True)

ranked_df.head(10)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,score,reason_code,action
0,content_cf56e2e2e282,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,5125.0,33705.0,...,0.84,24.11,0.0,excellent,striking,down,-85.6,61678,stale_visible_lowrank,refresh_content
1,content_7368877ea310,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,2591.0,16498.0,...,3.66,42.99,0.0,excellent,page_3_5,down,-81.5,59472,stale_visible_lowrank,refresh_content
2,content_1bfaa38ff26c,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3861.0,24672.0,...,3.75,43.33,0.0,good,page_3_5,down,-74.7,25715,stale_visible_lowrank,refresh_content
3,content_0a91db491d14,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3478.0,21948.0,...,5.13,41.76,0.0,good,striking,down,-51.8,13299,stale_visible_lowrank,refresh_content
4,content_5feee3994adb,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,transactional,3590.0,22780.0,...,0.00,40.00,0.0,good,page_3_5,down,-89.1,7812,stale_visible_lowrank,refresh_content
5,content_c2d929d83eaa,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,4758.0,30070.0,...,0.00,40.00,0.0,good,striking,down,-62.8,7558,stale_visible_lowrank,refresh_content
6,content_b16bd7307b39,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,4329.0,27844.0,...,0.00,25.00,0.0,good,page_3_5,down,-69.7,4590,stale_visible_lowrank,refresh_content
7,content_fe16a55cd13d,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3388.0,21742.0,...,2.38,38.64,0.0,good,striking,down,-52.2,4556,stale_visible_lowrank,refresh_content
8,content_ecb6215e79fd,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,4486.0,29333.0,...,25.00,33.33,0.0,good,page_3_5,down,-74.4,4429,stale_visible_lowrank,refresh_content
9,content_928af3e22c80,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3118.0,20396.0,...,0.00,0.00,0.0,moderate,striking,down,-45.7,1697,stale_visible_lowrank,refresh_content


In [8]:
import os

os.makedirs("work/outputs", exist_ok=True)
ranked_df.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Saved successfully!")

Saved successfully!


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [10]:
top20 = ranked_df[
    [
        "content_id",
        "score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d",
        "avg_position"
    ]
].head(20)

top20

,content_id,score,reason_code,action,days_since_last_update,impressions_90d,avg_position
0,content_cf56e2e2e282,61678,stale_visible_lowrank,refresh_content,194,61678,19.7
1,content_7368877ea310,59472,stale_visible_lowrank,refresh_content,194,59472,24.8
2,content_1bfaa38ff26c,25715,stale_visible_lowrank,refresh_content,194,25715,22.2
3,content_0a91db491d14,13299,stale_visible_lowrank,refresh_content,193,13299,10.5
4,content_5feee3994adb,7812,stale_visible_lowrank,refresh_content,194,7812,39.0
5,content_c2d929d83eaa,7558,stale_visible_lowrank,refresh_content,193,7558,17.9
6,content_b16bd7307b39,4590,stale_visible_lowrank,refresh_content,194,4590,31.0
7,content_fe16a55cd13d,4556,stale_visible_lowrank,refresh_content,194,4556,16.4
8,content_ecb6215e79fd,4429,stale_visible_lowrank,refresh_content,194,4429,25.3
9,content_928af3e22c80,1697,stale_visible_lowrank,refresh_content,193,1697,15.8


## Top-20 Review

| Rank | Action | Reason Code | Confidence Note | What would make it wrong? |
|------|--------|-------------|-----------------|---------------------------|
|1|Refresh Content|stale_visible_lowrank|Very high confidence. Very high impressions (61,678), 194 days since last update, and poor average position (19.7).|If the page is intentionally not being updated or traffic is highly seasonal.|
|2|Refresh Content|stale_visible_lowrank|Very high confidence. High impressions (59,472) and average position of 24.8 indicate strong improvement potential.|The ranking issue may be caused by strong competitors rather than outdated content.|
|3|Refresh Content|stale_visible_lowrank|High confidence. High visibility and old content make it a strong refresh candidate.|The page may already be scheduled for an update.|
|4|Refresh Content|stale_visible_lowrank|High confidence. Meets all baseline rule conditions with good visibility.|The page is close to page one (position 10.5), so improvements may be limited.|
|5|Refresh Content|stale_visible_lowrank|High confidence. Very poor ranking (39.0) despite good impressions.|Search intent may have changed and content refresh alone may not help.|
|6|Refresh Content|stale_visible_lowrank|High confidence. Old content with consistent impressions.|External ranking factors such as backlinks may be the real issue.|
|7|Refresh Content|stale_visible_lowrank|Moderate confidence. Meets all rule conditions.|Traffic could be temporary or seasonal.|
|8|Refresh Content|stale_visible_lowrank|Moderate confidence. Refresh may improve rankings.|The page may already satisfy user intent.|
|9|Refresh Content|stale_visible_lowrank|Moderate confidence. High opportunity because of low ranking.|Competition rather than content quality may limit ranking.|
|10|Refresh Content|stale_visible_lowrank|Moderate confidence. Rule identifies it as a refresh candidate.|Recent changes may not yet be reflected in search data.|
|11|Refresh Content|stale_visible_lowrank|Moderate confidence. Meets all baseline thresholds.|Manual review may decide the page is already optimized.|
|12|Refresh Content|stale_visible_lowrank|Moderate confidence. Low ranking and stale content justify review.|Business value of the page may be low.|
|13|Refresh Content|stale_visible_lowrank|Moderate confidence. Barely passes the impressions threshold but satisfies all conditions.|The small number of impressions may not justify immediate action.|
|14|Refresh Content|stale_visible_lowrank|Moderate confidence. Very poor ranking (48.0) suggests improvement opportunity.|Low search demand may limit the benefit of refreshing.|
|15|Keep Monitoring|no_action|High confidence. Does not satisfy the baseline rule because it is not stale enough.|The page may become stale in the future and require a refresh.|
|16|Keep Monitoring|no_action|High confidence. Not old enough despite reasonable impressions.|Performance should be monitored over time before acting.|
|17|Keep Monitoring|no_action|High confidence. Very recent update and very low impressions.|The page is too new to evaluate fairly.|
|18|Keep Monitoring|no_action|High confidence. Recently updated with almost no search traffic.|Traffic may increase as the page ages.|
|19|Keep Monitoring|no_action|High confidence. Not stale enough according to the rule.|Future performance may justify a refresh later.|
|20|Keep Monitoring|no_action|High confidence. High impressions but recently updated, so refresh is not recommended yet.|If rankings continue to stay low over time, the decision may change.|

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

The baseline rule is intentionally simple and may produce some incorrect recommendations.

### Potential Weak Picks

- Pages with high impressions but strong competition may not improve significantly after a content refresh.
- Some pages may perform poorly because of external factors such as backlinks or domain authority rather than outdated content.
- Pages with low search demand may receive a refresh recommendation even though the potential traffic gain is limited.
- Recently updated pages are excluded by the rule, but some of them may still require optimization for reasons unrelated to content freshness.

### Leakage Check

The baseline rule only uses the following input signals:

- `days_since_last_update`
- `impressions_90d`
- `avg_position`

The following columns were **not** used because they could introduce leakage or represent future information:

- `trend_direction`
- `trend_pct`
- any product flags
- any manually assigned labels
- any future-window information

Therefore, the baseline score is based only on historical, observable features and does not use label-derived or future information.

## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.